In [1]:
import pandas as pd
import os

In [2]:
paths = [
    ("datasets", "./data/Datasets.csv"),
    ("versions", "./data/DatasetVersions.csv"),
    ("dataset_tags", "./data/DatasetTags.csv"),
    ("tags", "./data/Tags.csv"),
]

for name, path in paths:
    df = pd.read_csv(path, nrows=3, low_memory=False)  # 仅读前3行，避免占内存
    print(f"\n== {name} | {path} ==")                  # 标注当前查看的表
    print("columns:", df.columns.tolist())             # 查看列名以识别主键/外键
    print(df.head(2))                                  # 打印两行样例，确认 id/datasetId/tagId 等


== datasets | ./data/Datasets.csv ==
columns: ['Id', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate']
       Id  CreatorUserId  OwnerUserId  OwnerOrganizationId  \
0  402034        2188609      2188609                  NaN   
1  402031        3792299      3792299                  NaN   

   CurrentDatasetVersionId  CurrentDatasourceVersionId  ForumId     Type  \
0                   771250                      792478   414197  Dataset   
1                   771247                      792475   414194  Dataset   

          CreationDate LastActivityDate  TotalViews  TotalDownloads  \
0  10/31/2019 01:07:32       10/31/2019        2706             128   
1  10/31/2019 00:58:15       10/31/2019        1795              37   

   TotalVotes  TotalKernels  Medal  MedalAwardDate  
0      

In [3]:
# 阶段 2a：检查 Datasets.csv 的版本指针质量（不做合并）
# 仅读取与检查有关的列，减小内存
ds = pd.read_csv("./data/Datasets.csv",
                 usecols=["Id", "CurrentDatasetVersionId", "CreationDate", "LastActivityDate"],
                 low_memory=False)

# 1) 规模与唯一性：Id 应与行数一致；若有重复说明主键异常
print("rows:", len(ds))                                  # 总行数
print("unique dataset Ids:", ds["Id"].nunique())         # 唯一的 Id 数
print("duplicate Ids:", ds["Id"].duplicated().sum())     # 统计重复 Id 的个数（期望为 0）

# 2) 当前版本指针缺失：若 >0，后续合并会出现 NaN 版本信息
print("null CurrentDatasetVersionId:",
      ds["CurrentDatasetVersionId"].isna().sum())

# 3) 当前版本指针复用：若 >0，表示有多个数据集指向同一 VersionId（通常不应发生）
print("duplicated CurrentDatasetVersionId:",
      ds["CurrentDatasetVersionId"].duplicated().sum())

# 4) 粗略时间一致性：创建时间不应晚于最近活动时间；>0 说明有异常时间记录
print("creation > last_activity count:",
      (pd.to_datetime(ds["CreationDate"], errors="coerce") >
       pd.to_datetime(ds["LastActivityDate"], errors="coerce")).sum())


rows: 521735
unique dataset Ids: 521735
duplicate Ids: 0
null CurrentDatasetVersionId: 245
duplicated CurrentDatasetVersionId: 244
creation > last_activity count: 517804


In [4]:
# 阶段 3｜标签映射与聚合（不引入行放大）
# 目标：
# 1) 用 Tags.csv 把 DatasetTags.csv 的 TagId 映射为可读的 TagName
# 2) 对每个 DatasetId 去重并聚合为一行（逗号分隔的标签字符串）
dt = pd.read_csv("./data/DatasetTags.csv", usecols=["DatasetId", "TagId"], low_memory=False)   # 只取必要列
tg = pd.read_csv("./data/Tags.csv", usecols=["Id", "Name"], low_memory=False)                  # 标签主表

tg = tg.rename(columns={"Id": "TagId", "Name": "TagName"})  # 对齐键名，便于合并
dt_named = dt.merge(tg, how="left", on="TagId")             # 映射 TagId -> TagName（左连接，保留所有打过标签的行）

dt_named = dt_named.drop_duplicates(subset=["DatasetId", "TagName"])  # 同一数据集重复的同名标签去重
tags_agg = (dt_named
            .groupby("DatasetId")["TagName"]                # 分组到每个数据集
            .apply(lambda s: ", ".join(sorted(s.dropna().astype(str))))  # 排序、去空、拼成逗号分隔
            .reset_index()
            .rename(columns={"TagName": "Tags"}))           # 得到：DatasetId, Tags

print(dt.shape, tg.shape, dt_named.shape, tags_agg.shape)   # 基本规模检查：聚合后行数应≈数据集中有标签的 DatasetId 数
print(tags_agg.head(3))                                     # 预览三行，确认 "DatasetId, Tags" 结构


(446054, 2) (831, 2) (446054, 3) (214603, 2)
   DatasetId                                            Tags
0          6  computer science, demographics, social science
1          7       internet, linguistics, online communities
2          8                atmospheric science, environment


In [5]:
# 将每个数据集的一行标签串（tags_agg）合并回 Datasets
ds = pd.read_csv("./data/Datasets.csv", low_memory=False)                     # 作为主表（保持一数据集一行）
ds_with_tags = ds.merge(tags_agg, how="left", left_on="Id", right_on="DatasetId")  # 按主键对齐，不引入行数放大
ds_with_tags = ds_with_tags.drop(columns=["DatasetId"])                       # 删除重复键列，避免“同样的列”

print("before/after shapes:", ds.shape, ds_with_tags.shape)                   # 行数应保持不变；列数多一列 Tags
print(ds_with_tags[["Id","Tags"]].head(3))                                    # 快速确认标签是否并入


before/after shapes: (521735, 16) (521735, 17)
       Id               Tags
0  402034                NaN
1  402031  health conditions
2   39875           bigquery


In [6]:
# 阶段 5｜填充缺失标签并保存结果（当前版本表已跳过）
# - 把没有标签的数据集的 Tags 填为空字符串，便于后续处理/检索
# - 将合并结果导出为 CSV 与 Parquet（若目录不存在则创建）
ds_with_tags["Tags"] = ds_with_tags["Tags"].fillna("")       # 将缺失标签填为空串，避免后续报错
ds_with_tags.to_csv("./data/metadata_merged_raw.csv", index=False)

In [9]:
# 阶段｜把 DatasetVersions 中的 Title/Slug 拼接到融合表，并把 Title/Slug 放到第2/3列
# - 从原始融合文件读取（你之前的文件名为 matadata_merged_raw.csv）
# - 仅读取版本表中合并所需的 3 列（Id/Title/Slug），用 python 引擎跳过坏行
# - 将版本表 Id 重命名为 CurrentDatasetVersionId，按同名键 left join（不改变行数）
# - 调整列顺序：保证 Title、Slug 分别为第2/第3列（Id 仍为第1列）
# - 覆盖保存为 ./data/metadata_merged_raw.csv

import pandas as pd

df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)  # 读取已融合的原始文件

dv = pd.read_csv(
    "./data/DatasetVersions.csv",
    usecols=["Id", "Title", "Slug"],   # 只读三列，减少解析压力
    engine="python",                   # python 引擎更耐脏数据
    on_bad_lines="skip"                # 跳过无法解析的坏行
)

print("dv shape:", dv.shape)           # 确认读取到的行列规模
print("dv columns:", dv.columns.tolist())
print(dv.head(3))   

dv shape: (1341929, 3)
dv columns: ['Id', 'Title', 'Slug']
   Id                               Title                                Slug
0  13                       US Baby Names                       us-baby-names
1  14                         SF Salaries                         sf-salaries
2  16  First GOP Debate Twitter Sentiment  first-gop-debate-twitter-sentiment


In [11]:
# 阶段：清洗 dv 的 Id 列（强制数值化 → 丢弃坏行 → 去重）
# 作用：
# - 把混入文本的 Id 强制转为数值；解析失败的行置为 NaN 后丢弃
# - 去重确保 1 个 VersionId 只保留 1 行，避免后续合并时异常放大

import pandas as pd

before = dv.shape[0]
dv["Id"] = pd.to_numeric(dv["Id"], errors="coerce")     # 非数字变 NaN
dv = dv.dropna(subset=["Id"])                           # 丢弃无法解析的行
dv["Id"] = dv["Id"].astype("Int64")                     # 转为可空整型，匹配主表的键类型
dv = dv.drop_duplicates(subset=["Id"], keep="last")     # 同一版本 ID 只保留一行

print("dv cleaned rows:", before, "->", dv.shape[0])     # 查看清洗后剩余行数
print(dv.head(3))                                       # 快速预览，确认 Id/Title/Slug 正常


dv cleaned rows: 1341929 -> 1331682
   Id                               Title                                Slug
0  13                       US Baby Names                       us-baby-names
1  14                         SF Salaries                         sf-salaries
2  16  First GOP Debate Twitter Sentiment  first-gop-debate-twitter-sentiment


In [12]:
# 阶段：把 Title/Slug 并入并放到第2/3列，然后覆盖保存 metadata_merged_raw.csv
# - 读取主表；若已存在 Title/Slug（上次中断残留），先删掉避免重复列
# - 对齐连接键类型为 Int64；用清洗后的 dv（含 Id/Title/Slug）改名后左连接
# - 调整列顺序：Id 保持第1列，Title/Slug 成为第2/3列；最后覆盖保存

import pandas as pd

df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)        # 读取主表
df = df.drop(columns=["Title", "Slug"], errors="ignore")                     # 防止历史残留导致重复列
df["CurrentDatasetVersionId"] = df["CurrentDatasetVersionId"].astype("Int64")  # 连接键统一为可空整型

dv_ = dv.rename(columns={"Id": "CurrentDatasetVersionId"})[["CurrentDatasetVersionId","Title","Slug"]].copy()
dv_["CurrentDatasetVersionId"] = dv_["CurrentDatasetVersionId"].astype("Int64")  # 与主表键类型一致

df = df.merge(dv_, how="left", on="CurrentDatasetVersionId")                 # 左连接：添加 Title/Slug（不改变行数）

rest = [c for c in df.columns if c not in ["Id","Title","Slug"]]             # 其余列
df = df[["Id","Title","Slug"] + rest]                                        # 调整顺序：Title/Slug 放到第2/3列

df.to_csv("./data/metadata_merged_raw.csv", index=False)                     # 覆盖保存
print("saved ./data/metadata_merged_raw.csv", df.shape, "first cols:", df.columns[:6].tolist())


saved ./data/metadata_merged_raw.csv (521735, 19) first cols: ['Id', 'Title', 'Slug', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId']


In [13]:
# 将“Tags”列移动到第4列；按 Id 升序排序；覆盖保存
import pandas as pd

df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)          # 读取当前文件
cols = ["Id", "Title", "Slug", "Tags"] + [c for c in df.columns if c not in ["Id","Title","Slug","Tags"]]  # 重排列顺序
df = df[cols].sort_values("Id", ascending=True).reset_index(drop=True)        # 按 Id 升序并重置索引
df.to_csv("./data/metadata_merged_raw.csv", index=False)                      # 覆盖保存
print("saved ./data/metadata_merged_raw.csv", df.shape, "first cols:", df.columns[:6].tolist())


saved ./data/metadata_merged_raw.csv (521735, 19) first cols: ['Id', 'Title', 'Slug', 'Tags', 'CreatorUserId', 'OwnerUserId']


In [14]:
# 阶段：从 DatasetVersions 取“每个 DatasetId 的最大 VersionNumber”对应的 Subtitle/Description，
#      并左连接到主表（按 Datasets.Id 对齐），把 Subtitle/Description 放在文件的最后两列，再覆盖保存

import pandas as pd

# 1) 读取主表与版本表（版本表仅取所需列；用 python 引擎并跳过坏行以容错长文本）
df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)
dv = pd.read_csv("./data/DatasetVersions.csv",
                 usecols=["DatasetId","VersionNumber","Subtitle","Description"],
                 engine="python", on_bad_lines="skip")

# 2) 将键与版本号转为数值；去掉无法解析的行（确保后续“取最大版本号”正确）
dv["DatasetId"] = pd.to_numeric(dv["DatasetId"], errors="coerce")
dv["VersionNumber"] = pd.to_numeric(dv["VersionNumber"], errors="coerce")
dv = dv.dropna(subset=["DatasetId","VersionNumber"])

# 3) 对每个 DatasetId 选取 VersionNumber 最大的那一行（即“最新版本”的副标题与描述）
idx = dv.groupby("DatasetId")["VersionNumber"].idxmax()
dv_latest = dv.loc[idx, ["DatasetId","Subtitle","Description"]].copy()

# 4) 左连接到主表：df.Id ↔ dv_latest.DatasetId；合并后删除冗余的 DatasetId 列
df = df.merge(dv_latest, how="left", left_on="Id", right_on="DatasetId")
df = df.drop(columns=["DatasetId"])

# 5) 将 Subtitle/Description 放到最后两列（若本就位于末尾，此步等价于显式确认顺序）
tail_cols = ["Subtitle","Description"]
df = df[[c for c in df.columns if c not in tail_cols] + tail_cols]

# 6) 覆盖保存
df.to_csv("./data/metadata_merged_raw.csv", index=False)
print("saved ./data/metadata_merged_raw.csv", df.shape, "last cols:", df.columns[-2:].tolist())


saved ./data/metadata_merged_raw.csv (521735, 21) last cols: ['Subtitle', 'Description']


In [15]:
# 阶段：识别“日期样式”的 object 列（≥90% 的值能被解析为日期）
import pandas as pd

df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)

obj_cols = [c for c in df.columns if df[c].dtype == "object"]  # 只看字符串类型的列
date_like = []
for c in obj_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() >= 0.90:   # 可解析率≥90% 就当作日期/时间列
        date_like.append(c)

print("date_like columns:", date_like)  # 这些列写出时将不加引号
text_cols = [c for c in obj_cols if c not in date_like]
print("text columns to quote (preview):", text_cols[:10], "… total:", len(text_cols))


/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/1719256981.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/1719256981.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/1719256981.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-e

date_like columns: ['CreationDate', 'LastActivityDate']
text columns to quote (preview): ['Title', 'Slug', 'Tags', 'Type', 'MedalAwardDate', 'Subtitle', 'Description'] … total: 7


In [16]:
# 阶段｜仅给“文本列”加双引号，日期列不加；覆盖保存到同一文件
# 思路：
# 1) 读取文件，自动识别 object 列中的“日期样式”列（≥90% 可被解析为日期）
# 2) 其余 object 列视为“文本列”；对这些列做 CSV 式双引号包裹（内部引号转成两连引号）
# 3) 数值列与日期列保持不加引号；逐行手写 CSV，最后覆盖保存

import pandas as pd
import numpy as np

# 读取现有文件
df = pd.read_csv("./data/metadata_merged_raw.csv", low_memory=False)

# 识别对象列与“日期样式”列（≥90% 可被解析为日期）
obj_cols = [c for c in df.columns if df[c].dtype == "object"]
date_like = []
for c in obj_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() >= 0.90:
        date_like.append(c)
text_cols = [c for c in obj_cols if c not in date_like]  # 仅这些列需要强制加双引号

# 手写 CSV：文本列强制双引号；其他列不加引号
with open("./data/metadata_merged_raw.csv", "w", encoding="utf-8", newline="") as f:
    # 写表头（列名不加引号）
    f.write(",".join(df.columns) + "\n")
    # 逐行写数据
    for row in df.itertuples(index=False, name=None):
        fields = []
        for col, val in zip(df.columns, row):
            if col in text_cols:
                # 文本列：空值写成""；非空先把内部 " 变成 ""，再整体用 " 包裹
                if pd.isna(val):
                    fields.append('""')
                else:
                    s = str(val).replace('"', '""')
                    fields.append(f'"{s}"')
            else:
                # 非文本列（数值/日期等）：空值写空串，非空直接字符串化（不加引号）
                fields.append("" if pd.isna(val) else str(val))
        f.write(",".join(fields) + "\n")

print("saved with quoted text columns (dates unquoted): ./data/metadata_merged_raw.csv")


/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/2279013788.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/2279013788.py:17: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_72911/2279013788.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and a

saved with quoted text columns (dates unquoted): ./data/metadata_merged_raw.csv
